In [1]:
from pathlib import Path
import csv
import unicodedata

import pandas as pd
from scipy.stats import spearmanr

from common_generate_predictions import load_data
from common_extra_processing import build_id

In [2]:
path_to_gold_data_spanish = "../dwug_es_cleaned"
gold_data_spanish = pd.read_csv("../test_data_es.csv", sep="\t")
gold_graded_change_data_spanish = gold_data_spanish.set_index("word")[
    "change_graded"
].to_dict()
print(gold_graded_change_data_spanish)
target_words_spanish = gold_data_spanish.word.tolist()
print(len(target_words_spanish))

{'actitud': 0.924660694787462, 'ataque': 0.481434326015839, 'atrás': 0.159569586509122, 'ausencia': 0.0, 'avance': 0.590113006548187, 'banco': 0.924660694787462, 'canal': 1.0, 'capital': 0.446772089340575, 'cobrar': 0.217259009452276, 'colaborar': 0.557923045284144, 'cólera': 0.740806952380577, 'compasión': 0.0, 'copiar': 0.459106313807781, 'corriente': 0.753008193531393, 'declinar': 0.572362532222144, 'demá': 0.319381864104736, 'diligencia': 0.469564643718015, 'disco': 0.914749624036807, 'distribuir': 0.0, 'educado': 0.159569586509122, 'elocuente': 0.0, 'encargado': 0.227813872100264, 'enterar': 0.0, 'especulación': 0.185296689786368, 'fallar': 0.171505575857383, 'fallecimiento': 0.0, 'historia': 0.0, 'historiador': 0.0, 'impulso': 0.159569586509122, 'indicativo': 1.0, 'juguete': 0.246987376051603, 'maduro': 0.19849916170588, 'maravilloso': 0.0, 'marco': 1.0, 'matiz': 0.0, 'médula': 0.295793409293228, 'metal': 0.557923045284144, 'metro': 0.469476812293811, 'modificado': 0.449423861592

In [3]:
path_to_gold_data_english = "../dwug_en"
with open(f"{path_to_gold_data_english}/target_words.txt", "r") as f_in:
    target_words_english = f_in.read().split("\n")

gold_data_english = pd.read_csv("../test_data_en.csv", sep="\t")

mask = gold_data_english["lemma"].isin(target_words_english)
gold_data_english = gold_data_english[mask]
gold_graded_change_data_english = gold_data_english.set_index("lemma")[
    "change_graded"
].to_dict()
print(len(gold_graded_change_data_english.keys()))
print(gold_data_english.shape)

37
(37, 19)


In [14]:
gold_compare_score_spanish = gold_data_spanish.set_index("word")["COMPARE"].to_dict()
gold_compare_score_spanish

GOLD_COMPARE_SCORE = {"dwug_es": gold_compare_score_spanish}

In [4]:
path_to_xl_lexeme_data = "../input/xl-lexeme/{dataset}/wic1/test.{tw}.scores"

In [5]:
gold_data_english.shape

(37, 19)

In [6]:
TARGET_WORDS = {"dwug_es": target_words_spanish, "dwug_en": target_words_english}
GOLD_DATA = {"dwug_es": gold_data_spanish, "dwug_en": gold_data_english}
PATH_TO_DATA = {
    "dwug_es": path_to_gold_data_spanish,
    "dwug_en": path_to_gold_data_english,
}
GOLD_GRADED_CHANGE_DATA = {
    "dwug_es": gold_graded_change_data_spanish,
    "dwug_en": gold_graded_change_data_english,
}

In [7]:
def process(dataset: str, target: str, uses: pd.DataFrame):
    try:
        clusters = pd.read_csv(
            f"{PATH_TO_DATA[dataset]}/clusters/opt/{target}.csv", sep="\t"
        )
    except Exception as e:
        clusters = pd.read_csv(
            f"{PATH_TO_DATA[dataset]}/clusters/{target}.csv", sep="\t"
        )
    uses_to_remove = clusters[clusters["cluster"] == -1].identifier.to_list()
    uses = uses[~uses.identifier.isin(uses_to_remove)]

    return uses

In [8]:
for d in ["dwug_es", "dwug_en"]:
    target_words = TARGET_WORDS[d]
    gold_data = GOLD_DATA[d]

    v1, v2 = [], []
    for tw in target_words:
        tw_p = unicodedata.normalize("NFC", tw)
        uses = pd.read_csv(
            f"{PATH_TO_DATA[d]}/data/{tw_p}/uses.csv",
            sep="\t",
            engine="python",
            quoting=csv.QUOTE_NONE if d == "dwug_en" else csv.QUOTE_MINIMAL,
        )
        judgments = pd.read_csv(
            f"{PATH_TO_DATA[d]}/data/{tw_p}/judgments.csv", sep="\t"
        )
        judgments = judgments[["identifier1", "identifier2", "judgment", "lemma"]]

        uses = process(d, tw_p, uses)
        uses = uses[["identifier", "context", "grouping"]]

        df = judgments.merge(uses, left_on=["identifier1"], right_on=["identifier"])
        del df["identifier"]

        df = df.rename(columns={"context": "context1", "grouping": "grouping1"})

        df = df.merge(uses, left_on=["identifier2"], right_on=["identifier"])
        del df["identifier"]
        df = df.rename(columns={"context": "context2", "grouping": "grouping2"})

        df = df.drop_duplicates(subset=["identifier1", "identifier2"])

        df["pair"] = df.apply(
            lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])), axis=1
        )

        q = path_to_xl_lexeme_data.format(dataset=d, tw=tw)
        xl_lexeme_data = pd.read_json(q)
        xl_lexeme_data["pair"] = xl_lexeme_data.apply(
            lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])), axis=1
        )

        new_data = df.merge(xl_lexeme_data, on=["pair"])
        new_data.drop(columns=["identifier1_y", "identifier2_y"], inplace=True)
        new_data.rename(
            columns={"identifier1_x": "identifier1", "identifier2_x": "identifier2"},
            inplace=True,
        )

        if d == "dwug_en":
            new_data = new_data.apply(lambda row: build_id(row), axis=1)

        mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
            "identifier2"
        ].str.startswith("old")
        mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
            "identifier1"
        ].str.startswith("old")

        new_data = new_data[mask1 | mask2]

        v1.append(GOLD_GRADED_CHANGE_DATA[d][tw])
        v2.append(new_data["score"].mean() * -1)

    assert len(v1) == len(v2), "Mismatch in vector's lenght"

    spr, _ = spearmanr(v1, v2)
    print(f"{d}: ", spr)

dwug_es:  0.5885593199830206
dwug_en:  0.8961347181059132


In [9]:
path_to_deepmistake_data = "../input/wic-scores/{dataset}/{model}.scores"

In [10]:
for d in ["dwug_es", "dwug_en"]:
    target_words = TARGET_WORDS[d]
    gold_data = GOLD_DATA[d]

    v1, v2 = [], []
    for tw in target_words:
        tw_p = unicodedata.normalize("NFC", tw)
        uses = pd.read_csv(
            f"{PATH_TO_DATA[d]}/data/{tw_p}/uses.csv",
            sep="\t",
            engine="python",
            quoting=csv.QUOTE_NONE if d == "dwug_en" else csv.QUOTE_MINIMAL,
        )
        judgments = pd.read_csv(
            f"{PATH_TO_DATA[d]}/data/{tw_p}/judgments.csv", sep="\t"
        )
        judgments = judgments[["identifier1", "identifier2", "judgment", "lemma"]]

        uses = process(d, tw_p, uses)
        uses = uses[["identifier", "context", "grouping"]]

        df = judgments.merge(uses, left_on=["identifier1"], right_on=["identifier"])
        del df["identifier"]

        df = df.rename(columns={"context": "context1", "grouping": "grouping1"})

        df = df.merge(uses, left_on=["identifier2"], right_on=["identifier"])
        del df["identifier"]
        df = df.rename(columns={"context": "context2", "grouping": "grouping2"})

        df = df.drop_duplicates(subset=["identifier1", "identifier2"])

        df["pair"] = df.apply(
            lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])), axis=1
        )

        q = path_to_deepmistake_data.format(dataset=d, model="wic6")
        deepmistake_data = pd.read_csv(q)
        deepmistake_data = deepmistake_data[deepmistake_data["word"] == tw]
        deepmistake_data["pair"] = deepmistake_data.apply(
            lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])), axis=1
        )

        new_data = df.merge(deepmistake_data, on=["pair"])
        new_data.drop(columns=["identifier1_y", "identifier2_y"], inplace=True)
        new_data.rename(
            columns={"identifier1_x": "identifier1", "identifier2_x": "identifier2"},
            inplace=True,
        )

        if d == "dwug_en":
            new_data = new_data.apply(lambda row: build_id(row), axis=1)

        mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
            "identifier2"
        ].str.startswith("old")
        mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
            "identifier1"
        ].str.startswith("old")

        new_data = new_data[mask1 | mask2]

        v1.append(GOLD_GRADED_CHANGE_DATA[d][tw])
        v2.append(new_data["score"].mean() * -1)

    assert len(v1) == len(v2), "Mismatch in vector's lenght"

    spr, _ = spearmanr(v1, v2)
    print(f"{d}: ", spr)

dwug_es:  0.6949234296422077
dwug_en:  0.8828551350908481


In [11]:
for model in ["xl_lexeme", "deepmistake"]:
    print(model)

    for d in ["dwug_es", "dwug_en"]:
        print(f"  {d}", end=": ")
        target_words = TARGET_WORDS[d]

        v1, v2 = [], []

        for tw in target_words:
            tw_p = unicodedata.normalize("NFC", tw)
            uses = pd.read_csv(
                f"{PATH_TO_DATA[d]}/data/{tw_p}/uses.csv",
                sep="\t",
                engine="python",
                quoting=csv.QUOTE_NONE if d == "dwug_en" else csv.QUOTE_MINIMAL,
            )
            judgments = pd.read_csv(
                f"{PATH_TO_DATA[d]}/data/{tw_p}/judgments.csv", sep="\t"
            )
            judgments = judgments[["identifier1", "identifier2", "judgment", "lemma"]]

            uses = process(d, tw_p, uses)
            uses = uses[["identifier", "context", "grouping"]]

            df = judgments.merge(uses, left_on=["identifier1"], right_on=["identifier"])
            del df["identifier"]

            df = df.rename(columns={"context": "context1", "grouping": "grouping1"})

            df = df.merge(uses, left_on=["identifier2"], right_on=["identifier"])
            del df["identifier"]
            df = df.rename(columns={"context": "context2", "grouping": "grouping2"})

            # df = df.drop_duplicates(subset=["identifier1", "identifier2"])

            df["pair"] = df.apply(
                lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                axis=1,
            )

            df_filtered_median = df.groupby("pair")["judgment"].mean().reset_index()

            if model == "xl_lexeme":
                q = path_to_xl_lexeme_data.format(dataset=d, tw=tw)
                data = pd.read_json(q)
            else:
                q = path_to_deepmistake_data.format(dataset=d, model="wic7")
                data = pd.read_csv(q)
                data = data[data["word"] == tw_p]

            data["pair"] = data.apply(
                lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                axis=1,
            )
            new_data = df_filtered_median.merge(data, on=["pair"])

            if d == "dwug_en":
                new_data = new_data.apply(lambda row: build_id(row), axis=1)

            mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
                "identifier2"
            ].str.startswith("old")
            mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
                "identifier1"
            ].str.startswith("old")

            new_data = new_data[mask1 | mask2]

            v1.extend(new_data["score"].tolist())
            v2.extend(new_data["judgment"].tolist())

        spr, _ = spearmanr(v1, v2)
        print(spr)

xl_lexeme
  dwug_es: 0.5612322822482348
  dwug_en: 0.6179364550754184
deepmistake
  dwug_es: 0.6212740722310739
  dwug_en: 0.6120928805601821


In [12]:
for model in ["xl_lexeme", "deepmistake"]:
    print(model)

    for d in ["dwug_es", "dwug_en"]:
        print(f"  {d}", end=": ")
        target_words = TARGET_WORDS[d]

        v1, v2 = [], []

        for tw in target_words:
            tw_p = unicodedata.normalize("NFC", tw)
            uses = pd.read_csv(
                f"{PATH_TO_DATA[d]}/data/{tw_p}/uses.csv",
                sep="\t",
                engine="python",
                quoting=csv.QUOTE_NONE if d == "dwug_en" else csv.QUOTE_MINIMAL,
            )
            judgments = pd.read_csv(
                f"{PATH_TO_DATA[d]}/data/{tw_p}/judgments.csv", sep="\t"
            )
            judgments = judgments[["identifier1", "identifier2", "judgment", "lemma"]]

            # uses = process(d, tw_p, uses)
            uses = uses[["identifier", "context", "grouping"]]

            df = judgments.merge(uses, left_on=["identifier1"], right_on=["identifier"])
            del df["identifier"]

            df = df.rename(columns={"context": "context1", "grouping": "grouping1"})

            df = df.merge(uses, left_on=["identifier2"], right_on=["identifier"])
            del df["identifier"]
            df = df.rename(columns={"context": "context2", "grouping": "grouping2"})

            # df = df.drop_duplicates(subset=["identifier1", "identifier2"])

            df["pair"] = df.apply(
                lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                axis=1,
            )

            df_filtered_median = df.groupby("pair")["judgment"].mean().reset_index()

            if model == "xl_lexeme":
                q = path_to_xl_lexeme_data.format(dataset=d, tw=tw)
                data = pd.read_json(q)
            else:
                q = path_to_deepmistake_data.format(dataset=d, model="wic7")
                data = pd.read_csv(q)
                data = data[data["word"] == tw_p]

            data["pair"] = data.apply(
                lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                axis=1,
            )
            new_data = df_filtered_median.merge(data, on=["pair"])

            if d == "dwug_en":
                new_data = new_data.apply(lambda row: build_id(row), axis=1)

            mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
                "identifier2"
            ].str.startswith("old")
            mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
                "identifier1"
            ].str.startswith("old")

            new_data = new_data[mask1 | mask2]

            v1.append(GOLD_GRADED_CHANGE_DATA[d][tw])
            v2.append(new_data["score"].mean() * (-1))

        assert len(v1) == len(v2), "mismatch vectors"

        spr, _ = spearmanr(v1, v2)
        print(spr)

xl_lexeme
  dwug_es: 0.5851010103113101
  dwug_en: 0.8975575305718131
deepmistake
  dwug_es: 0.6347825809610019
  dwug_en: 0.8674413333769332


In [31]:
for model in ["xl_lexeme", "deepmistake"]:
    print(model)

    for d in ["dwug_es"]:
        print(f"  {d}", end=": ")
        target_words = TARGET_WORDS[d]

        v1, v2 = [], []

        for tw in target_words:
            tw_p = unicodedata.normalize("NFC", tw)
            uses = pd.read_csv(
                f"{PATH_TO_DATA[d]}/data/{tw_p}/uses.csv",
                sep="\t",
                engine="python",
                quoting=csv.QUOTE_NONE if d == "dwug_en" else csv.QUOTE_MINIMAL,
            )
            judgments = pd.read_csv(
                f"{PATH_TO_DATA[d]}/data/{tw_p}/judgments.csv", sep="\t"
            )
            judgments = judgments[["identifier1", "identifier2", "judgment", "lemma"]]

            # uses = process(d, tw_p, uses)
            uses = uses[["identifier", "context", "grouping"]]

            df = judgments.merge(uses, left_on=["identifier1"], right_on=["identifier"])
            del df["identifier"]

            df = df.rename(columns={"context": "context1", "grouping": "grouping1"})

            df = df.merge(uses, left_on=["identifier2"], right_on=["identifier"])
            del df["identifier"]
            df = df.rename(columns={"context": "context2", "grouping": "grouping2"})

            # df = df.drop_duplicates(subset=["identifier1", "identifier2"])

            df["pair"] = df.apply(
                lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                axis=1,
            )

            df_filtered_median = df.groupby("pair")["judgment"].mean().reset_index()

            if model == "xl_lexeme":
                q = path_to_xl_lexeme_data.format(dataset=d, tw=tw)
                data = pd.read_json(q)
            else:
                q = path_to_deepmistake_data.format(dataset=d, model="wic7")
                data = pd.read_csv(q)
                data = data[data["word"] == tw_p]

            data["pair"] = data.apply(
                lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                axis=1,
            )
            new_data = df_filtered_median.merge(data, on=["pair"])

            if d == "dwug_en":
                new_data = new_data.apply(lambda row: build_id(row), axis=1)

            mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
                "identifier2"
            ].str.startswith("old")
            mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
                "identifier1"
            ].str.startswith("old")

            new_data = new_data[mask1 | mask2]

            v1.append(GOLD_COMPARE_SCORE[d][tw])
            v2.append(new_data["score"].mean())

        assert len(v1) == len(v2), "mismatch vectors"

        spr, _ = spearmanr(v1, v2)
        print(spr)

xl_lexeme
  dwug_es: 0.6925812725757156
deepmistake
  dwug_es: 0.7796610169491527
